## Set up

In [1]:
# ============================================================
# BACKTEST: Run SEPAHybridV1 from Notebook
# ============================================================
# Paste this into a notebook cell. It will:
# 1. Score the universe from T3 (daily, ~125K rows)
# 2. Run the full BackTrader backtest
# 3. Display QuantStats interactive tearsheet inline
# 4. Show trade log and summary stats
import sys, logging
from pathlib import Path
from datetime import datetime, timedelta

ROOT = Path().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL = 'm01_prototype_2003_2026' # 22 24
VERSION = "v1"
model_dir = ROOT / 'models' / MODEL / VERSION
model_path  = str(model_dir / 'model.json')
cal_path  = model_dir / 'calibration.json'
cal_str   = str(cal_path) if cal_path.exists() else None

logging.basicConfig(level=logging.WARNING,
                    format='%(asctime)s | %(levelname)s | %(message)s',
                    datefmt='%H:%M:%S')


In [2]:
# ============================================================
# BACKTEST HELPER
# ============================================================
import importlib
%load_ext autoreload
%autoreload 2

def reload_backtest():
    """Reload all backtest modules (call after editing source)."""
    import src.backtest.runner, src.backtest.sepa_strategy, src.backtest.universe_scorer
    importlib.reload(src.backtest.universe_scorer)
    importlib.reload(src.backtest.runner)
    importlib.reload(src.backtest.sepa_strategy)

reload_backtest()
from src.backtest.runner import SEPABacktestRunner
from src.backtest.sepa_strategy import SEPAHybridV1, SEPAFlatV1
from src.backtest.universe_scorer import UniverseScorer

def run_backtest(
    start: str,
    end: str,
    strategy=SEPAFlatV1,
    model: str = 'models/m01_prototype/model.json',
    cash: float = 100_000,
    warmup_months: int = 6,
    max_tickers: int = None,
    **strategy_overrides,
):
    """
    Run a full backtest: score → setup → run → report.
    
    Args:
        start: Eval start date 'YYYY-MM-DD' (BackTrader starts warmup_months earlier)
        end: End date 'YYYY-MM-DD'
        strategy: Strategy class (SEPAFlatV1, SEPAHybridV1, etc.)
        model: Path to model.json
        cash: Initial capital
        warmup_months: Months of warmup before start (for SMA50 etc.)
        max_tickers: Cap tickers for smoke testing
        **strategy_overrides: Any strategy params to override (e.g., min_prob_elite=0.30)
    
    Returns:
        (metrics, runner, trade_df)
    """
    eval_start = start
    bt_start = (datetime.strptime(start, '%Y-%m-%d') - timedelta(days=warmup_months * 30)).strftime('%Y-%m-%d')
    score_start = (datetime.strptime(bt_start, '%Y-%m-%d') - timedelta(days=365)).strftime('%Y-%m-%d')
    
    # 1. Score
    print(f"[1/3] Scoring ({score_start} → {end})...")
    scorer = UniverseScorer(m01_path=model)
    scores_df = scorer.score_from_t3(score_start, end)
    print(f"      {len(scores_df):,} rows, {scores_df['ticker'].nunique()} tickers")
    
    # 2. Setup
    print(f"[2/3] Setting up ({bt_start} → {end}, eval from {eval_start})...")
    runner = SEPABacktestRunner(start_date=bt_start, end_date=end, initial_cash=cash)
    runner.setup(scores_df=scores_df, max_tickers=max_tickers)
    
    runner.cerebro.strats = []
    runner.cerebro.addstrategy(strategy, scores_df=scores_df, **strategy_overrides)
    
    # 3. Run
    print(f"[3/3] Running backtest...")
    metrics = runner.run()
    runner.print_results(metrics)
    
    # 4. Trade analysis (eval window only)
    trade_df = runner.get_trade_dataframe()
    if trade_df is not None and len(trade_df) > 0:
        trade_df['entry_date'] = pd.to_datetime(trade_df['entry_date'])
        eval_trades = trade_df[trade_df['entry_date'] >= eval_start]
        
        if len(eval_trades) > 0:
            wins = eval_trades[eval_trades['pnl_percent'] > 0]
            losses = eval_trades[eval_trades['pnl_percent'] <= 0]
            pf = abs(wins['pnl_percent'].sum() / losses['pnl_percent'].sum()) if len(losses) and losses['pnl_percent'].sum() != 0 else float('inf')
            
            print(f"\n{'='*50}")
            print(f"EVAL WINDOW: {eval_start} → {end}")
            print(f"{'='*50}")
            print(f"  Trades:         {len(eval_trades)}")
            print(f"  Win Rate:       {len(wins)/len(eval_trades)*100:.1f}%")
            print(f"  Avg PnL:        {eval_trades['pnl_percent'].mean():+.2f}%")
            print(f"  Median PnL:     {eval_trades['pnl_percent'].median():+.2f}%")
            print(f"  Profit Factor:  {pf:.2f}")  

    return metrics, runner, trade_df


## Enrichment & Data Prep

## Run Backtest

In [3]:
# Basic run
# metrics, runner, trades = run_backtest('2020-01-01', '2025-01-01')

# Different date range
# metrics, runner, trades = run_backtest('2010-01-01', '2015-01-01')

# Different strategy
# metrics, runner, trades = run_backtest('2020-01-01', '2025-01-01', strategy=SEPAFlatV1, model=m01_path)

# Override specific params
# metrics, runner, trades = run_backtest(
#     '2020-01-01', '2025-01-01',
#     min_prob_elite=0.50,
#     max_stop_pct=0.20,
#     sma_exit_period=100,
#     model=model_path
# )

# Smoke test (fast)
# metrics, runner, trades = run_backtest('2024-01-01', '2024-06-30', max_tickers=50)

# 1. Clear any old strategies (crucial if running in a Jupyter cell multiple times)
# runner.cerebro.strats = []
# 2. Define the exact parameters you want to override for this run
SWEEP_BEST = {
    'min_prob_elite': 0.5, 
    'stop_loss_pct': 0.2,  
    'sma_exit_period': 100,
}
strategy_kwargs = {
 # Overrides from the vectorized sweep
    'min_prob_elite': SWEEP_BEST['min_prob_elite'],
    'max_stop_pct': SWEEP_BEST['stop_loss_pct'],
    'sma_exit_period': SWEEP_BEST['sma_exit_period'],
    
    # Static parameters you want to enforce
    'min_score': 30,               
    'rank_by': 'trailing',
    'min_price': 5.0,
    'cooldown_days': 3,
    'warmup_days': 10,
    'atr_stop_mult': 2.0,                           
    'atr_target1_mult': 3.0,       
    'min_target1_pct': 0.15,
    'atr_target2_add': 2.0,
    
    # Use Hybrid mode's regime sizing
    'sizing_mode': 'equal_weight',
    'regime_max_pos': {0:0, 1:10, 2:10, 3:10, 4:10},
    # 'regime_sizes': {0:0.0, 1:0.1, 2:0.1, 3:0.1, 4:0.10},
}

# 3. Add the strategy to cerebro
# runner.cerebro.addstrategy(SEPAHybridV1, **strategy_kwargs)

metrics, runner, trades = run_backtest('2020-01-01', '2025-12-31', strategy=SEPAHybridV1, model=model_path, **strategy_kwargs)

# QuantStats tearsheet after any run
import quantstats as qs

equity_df = runner.get_equity_curve_dataframe()
if equity_df is not None and len(equity_df) > 2:
    returns = equity_df['value'].pct_change().dropna()
    returns.index = pd.to_datetime(returns.index)
    returns.index.name = None  # QuantStats requirement
    print(f"\n{'='*60}")
    print("QUANTSTATS TEARSHEET")
    print(f"{'='*60}")
    qs.extend_pandas()
    # Show key metrics inline
    print(f"CAGR:           {qs.stats.cagr(returns)*100:.2f}%")
    print(f"Sharpe:         {qs.stats.sharpe(returns):.2f}")
    print(f"Sortino:        {qs.stats.sortino(returns):.2f}")
    print(f"Max Drawdown:   {qs.stats.max_drawdown(returns)*100:.2f}%")
    print(f"Calmar:         {qs.stats.calmar(returns):.2f}")
    print(f"Win Rate:       {qs.stats.win_rate(returns)*100:.1f}%")
    
    # Full interactive HTML tearsheet (renders in notebook)
    qs.reports.full(returns, benchmark='SPY', output=None, download_filename=None)


[1/3] Scoring (2018-07-05 → 2025-12-31)...


12:59:44 | WARNING | No calibration table at models\m01_calibration.json


XGBoostError: [12:59:45] C:\actions-runner\_work\xgboost\xgboost\src\data\../data/cat_container.h:28: Invalid new DataFrame input for the: 81th feature (0-based). The data type doesn't match the one used in the training dataset. Both should be either numeric or categorical. For a categorical feature, the index type must match between the training and test set.

## (Optional) Parameter Sweep

In [ ]:
import itertools
import pandas as pd
import quantstats as qs
from src.backtest.vectorized_backtest import VectorizedSEPABacktest

GRID = {
    'min_prob_elite':        [0.1, 0.5, 0.8],
    'stop_loss_pct':         [0.10, 0.15,0.2,0.25],
    'sma_exit_period':       [20, 100],
    'max_positions_per_day': [3, 8, 10],
}

FIXED = dict(
    model_path=model_path,
    start_date='2020-01-01',
    end_date='2025-01-01',
    ranking_lookback_days=10,
    warmup_days=10,
    initial_cash=100_000,
    position_size_pct=0.10,
    max_hold_days=252,
)

keys = list(GRID.keys())
combos = list(itertools.product(*GRID.values()))
print(f"Running {len(combos)} combos...\n")

results = []
for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    vbt = VectorizedSEPABacktest(**FIXED, **params)
    trades = vbt.run()
    
    if trades.empty or len(trades) < 5:
        results.append({**params, 'n_trades': len(trades), 
                        'total_return': 0, 'sharpe': 0, 'max_dd': 0,
                        'win_rate': 0, 'avg_pnl': 0, 'profit_factor': 0})
        continue
    
    # Trade-level stats
    wins = trades[trades['pnl_pct'] > 0]
    losses = trades[trades['pnl_pct'] <= 0]
    pf = abs(wins['pnl_pct'].sum() / losses['pnl_pct'].sum()) if len(losses) and losses['pnl_pct'].sum() != 0 else float('inf')
    
    # Portfolio-level stats via equity curve
    equity = vbt.equity_curve(trades)
    returns = equity.pct_change().dropna()
    
    total_ret = (equity.iloc[-1] - equity.iloc[0]) / equity.iloc[0] * 100
    sharpe = qs.stats.sharpe(returns) if len(returns) > 10 else 0
    max_dd = qs.stats.max_drawdown(returns) * 100 if len(returns) > 10 else 0
    
    row = {
        **params,
        'n_trades': len(trades),
        'win_rate': len(wins) / len(trades),
        'avg_pnl': trades['pnl_pct'].mean(),
        'profit_factor': pf,
        'total_return': total_ret,
        'sharpe': sharpe,
        'max_dd': max_dd,
    }
    results.append(row)
    print(f"[{i+1}/{len(combos)}] "
          f"elite={params['min_prob_elite']} stop={params['stop_loss_pct']} "
          f"sma={params['sma_exit_period']} pos={params['max_positions_per_day']}  →  "
          f"trades={len(trades):3d}  return={total_ret:+.1f}%  "
          f"sharpe={sharpe:.2f}  maxDD={max_dd:.1f}%  PF={pf:.2f}")

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('sharpe', ascending=False)

print(f"\n{'='*100}")
print("SWEEP RESULTS (sorted by Sharpe)")
print(f"{'='*100}")
display(results_df.style.format({
    'win_rate': '{:.1%}', 'avg_pnl': '{:+.2%}', 'profit_factor': '{:.2f}',
    'total_return': '{:+.1f}%', 'sharpe': '{:.2f}', 'max_dd': '{:.1f}%'
}))


In [ ]:
# ============================================================
# VALIDATE SWEEP WINNER IN BACKTRADER
# ============================================================
# Translates vectorized sweep params → BackTrader strategy params

import importlib
import src.backtest.runner, src.backtest.sepa_strategy
importlib.reload(src.backtest.runner)
importlib.reload(src.backtest.sepa_strategy)
from src.backtest.runner import SEPABacktestRunner
from src.backtest.sepa_strategy import SEPAHybridV1, SEPAFlatV1

# --- YOUR SWEEP WINNER ---
SWEEP_BEST = {
    'min_prob_elite': 0.5,    # From sweep
    'stop_loss_pct': 0.2,     # From sweep
    'sma_exit_period': 100,     # From sweep
}

# --- TRANSLATED TO BACKTRADER ---
BT_START     = '2019-06-01'   # 6-month warmup for SMA50
END_DATE     = '2025-01-01'
EVAL_START   = '2020-01-01'
INITIAL_CASH = 100_000

runner = SEPABacktestRunner(
    start_date=BT_START,
    end_date=END_DATE,
    initial_cash=INITIAL_CASH,
)
runner.setup(scores_df=scores_df)

# Override strategy with sweep-winner params
runner.cerebro.strats = []
runner.cerebro.addstrategy(
    SEPAFlatV1,
    scores_df=scores_df,
    
    # === ENTRY (from sweep) ===
    min_prob_elite=SWEEP_BEST['min_prob_elite'],
    min_score=30,               # Keep safety floor
    rank_by='trailing',
    min_price=5.0,
    cooldown_days=3,
    warmup_days=10,
    
    # === EXIT (bridged from sweep) ===
    max_stop_pct=SWEEP_BEST['stop_loss_pct'],   # Cap ATR stop at sweep's flat %
    atr_stop_mult=2.0,                           # Keep ATR-based, capped by above
    sma_exit_period=SWEEP_BEST['sma_exit_period'],
    
    # === TARGETS (BT-only, not in sweep) ===
    atr_target1_mult=3.0,       # Keep defaults
    min_target1_pct=0.15,
    atr_target2_add=2.0,
    
    # === SIZING (BT-only) ===
    sizing_mode='regime',
    regime_max_pos={0:0, 1:4, 2:8, 3:10, 4:12},
    regime_sizes={0:0.0, 1:0.025, 2:0.05, 3:0.075, 4:0.10},
)

print("Running BackTrader validation...")
metrics = runner.run()
runner.print_results(metrics)

# --- COMPARE WITH SWEEP ---
import pandas as pd
trade_df = runner.get_trade_dataframe()
if trade_df is not None:
    trade_df['entry_date'] = pd.to_datetime(trade_df['entry_date'])
    eval_trades = trade_df[trade_df['entry_date'] >= EVAL_START]
    
    if len(eval_trades) > 0:
        wins = eval_trades[eval_trades['pnl_percent'] > 0]
        losses = eval_trades[eval_trades['pnl_percent'] <= 0]
        
        print(f"\n{'='*60}")
        print(f"COMPARISON (eval window: {EVAL_START} onwards)")
        print(f"{'='*60}")
        print(f"{'Metric':<25} {'Vectorized':<15} {'BackTrader':<15}")
        print(f"{'-'*55}")
        print(f"{'Trades':<25} {'915':<15} {len(eval_trades):<15}")
        print(f"{'Win Rate':<25} {'33.0%':<15} {len(wins)/len(eval_trades)*100:.1f}%")
        print(f"{'Avg PnL/trade':<25} {'+4.09%':<15} {eval_trades['pnl_percent'].mean():+.2f}%")
        print(f"{'Median PnL':<25} {'—':<15} {eval_trades['pnl_percent'].median():+.2f}%")
        total_pnl = eval_trades['pnl_percent'].sum()
        print(f"{'Total PnL Sum':<25} {'—':<15} {total_pnl:+.0f}%")
        pf = abs(wins['pnl_percent'].sum() / losses['pnl_percent'].sum()) if len(losses) else float('inf')
        print(f"{'Profit Factor':<25} {'1.63':<15} {pf:.2f}")
        tr = metrics.get('total_return', 0)
        sr = metrics.get('sharpe_ratio', 'N/A')
        md = metrics.get('max_drawdown', 0)
        print(f"{'Portfolio Return':<25} {'—':<15} {tr:+.1f}%")
        print(f"{'Sharpe':<25} {'—':<15} {sr}")
        print(f"{'Max Drawdown':<25} {'—':<15} {md:.1f}%")

